# NB 04 · Conditional routing and loops

**Series:** Learning LangGraph through literate notebooks, notebook 4 of 7.
**Central question:** how does the assistant search again when the evidence is thin, without looping forever?

NB 03 ended with two questions stuck at the quality gate. For topic e, the correct figure (0.62%)
rests on a single source; for topic c, only the supervisory authority's circular states a figure.
A linear chain has one move left at that point: write the answer anyway. This notebook gives the
assistant a second move. After scoring, a router decides whether to answer, to search again with a
different query, or to stop and state what is missing.

## Learning goals

After this notebook you can:

- write a router function and connect it with `add_conditional_edges` and a complete map of its labels;
- build a loop that leads back to an earlier node, and explain why the state carries over between rounds;
- give a loop a business stop condition, and tell it apart from LangGraph's technical `recursion_limit`;
- make a repeated search search differently, and judge whether a second round helped;
- explain why a field that is written in every round needs a merge rule.

## Setup

The scripted model now combines two recordings from `qwen3:8b`: the claims from NB 03 and the search
queries written for this notebook. Everything NB 03 built (the collectors, the scorer, the writer and
the evidence rules) is imported from the package.

In [ ]:
import operator
import sys
from importlib.metadata import version
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph

from research_assistant.assistant import (
    RESEARCH_CONTRACT,
    build_research_graph,
    collect_primary,
    collect_secondary,
    corpus,
    extract_claim,
    make_score,
    weights,
    write_answer,
)
from research_assistant.evidence import AUTHORITY_FLOOR, CANONICAL_QUESTIONS, EVIDENCE_THRESHOLD, assess
from research_assistant.models import get_chat_model, get_llm_mode
from research_assistant.retrieval import retrieve
from research_assistant.state import ResearchState, SourceDoc, check_contract, merge_claims, merge_documents

load_dotenv()

assert sys.version_info >= (3, 12), "This series needs Python 3.12 or newer."
print(f"Python {sys.version.split()[0]}")
for package in ("langgraph", "langchain-core"):
    print(f"{package} {version(package)}")

mode = get_llm_mode()
if mode == "disabled":
    mode = "fake"  # this notebook needs a model; the recorded replies stand in
llm = get_chat_model(mode, fixture=("nb03_claims.yaml", "nb04_queries.yaml"))
print(f"Model mode: {mode}")
QUESTIONS = CANONICAL_QUESTIONS

## Concept: a loop with a purpose

A chain runs its steps once, in a fixed order. A graph can lead back: a **conditional edge** calls a
small function, the **router**, which reads the state and returns a label; a map turns the label into
the next node. With one edge from the router back to the search, the assistant can try again.

```text
          +--> [collect_primary] ----+
START ----+                          +--> [score] --router--+-- answer ------> [write_answer] -----> END
          +--> [collect_secondary] --+                      +-- limit_answer -> [write_limitation] -> END
                 ^            ^                             +-- search_again
                 |            |                                   |
                 +------------+------- [refine_query] <-----------+
```

A loop is easy to draw and easy to get wrong. A good one needs three things:

1. **a purpose driven by quality:** search again only because the quality gate failed, never by habit;
2. **state that accumulates safely:** what round 1 found must survive round 2, without duplicates;
3. **a stop condition from the domain:** at most three search rounds (P5 in ADR 0007), because
   another round over 21 documents will not find what two rounds missed.

LangGraph adds a fourth, technical safeguard of its own: `recursion_limit`, which aborts any run that
takes too many steps. It is an emergency brake, not a stop condition; mistake 1 shows the difference.

## Experiment

### 1 Where NB 03 stopped

The NB 03 graph, imported from the package, on the two questions that fail the gate:

In [ ]:
research_graph = build_research_graph(llm)
for topic in ("c", "e"):
    print(f"{topic}: {research_graph.invoke({'question': QUESTIONS[topic]})['draft_answer'][:110]}...")

### 2 The router

The router is a plain function. It reads two fields, the evidence score and the round counter, and
returns one of three labels. It writes nothing: deciding where to go is separate from changing the
state.

| Label | When | Next node |
|---|---|---|
| `answer` | the evidence score reaches the threshold of 1.0 | `write_answer` |
| `search_again` | the gate failed and fewer than 3 rounds have run | `refine_query` |
| `limit_answer` | the gate failed in round 3 | `write_limitation` |

The evidence score already encodes the full quality gate: it only counts independent sources above the
authority floor, and no single source exceeds 0.95, so a score of 1.0 or more always means at least
two of them.

In [ ]:
MAX_ROUNDS = 3  # P5: the business stop condition


def route(state: ResearchState) -> str:
    if state["evidence_score"] >= EVIDENCE_THRESHOLD:
        return "answer"
    if state.get("retrieval_round", 1) < MAX_ROUNDS:
        return "search_again"
    return "limit_answer"


for example in ({"evidence_score": 1.6}, {"evidence_score": 0.85}, {"evidence_score": 0.95, "retrieval_round": 3}):
    print(example, "->", route(example))

### 3 A second search must search differently

Running the same search again returns the same documents. Each round therefore gets its own
strategy, while the quality gate stays exactly the same:

| Round | Strategy |
|---|---|
| 1 | the words of the question |
| 2 | the model rephrases the question with synonyms and technical terms |
| 3 | the model writes a broader query about the general subject |

`refine_query` asks the model for the next query, raises the round counter and is the only node that
writes the two new fields, `query` and `retrieval_round`. The collectors search with `query` when it
is set. The prompt log explains how these two prompts came about.

In [ ]:
SYNONYM_PROMPT = ChatPromptTemplate.from_template(
    "Write a new search query for the question below. Use different words: replace the "
    "question's key terms with synonyms and the technical terms used in the field, so that "
    "documents that describe the same thing in other words are found too. Keep any names. "
    "Reply with the query only, on one line.\n\nQuestion: {question}"
)
BROADER_PROMPT = ChatPromptTemplate.from_template(
    "Write a broader search query about the general subject of the question below. Leave out "
    "details such as figures, dates and the exact measure asked for. Keep any names. "
    "Reply with the query only, on one line.\n\nQuestion: {question}"
)
QUERY_PROMPTS = {2: SYNONYM_PROMPT, 3: BROADER_PROMPT}


def clean_query(reply: str) -> str:
    line = next((line for line in reply.strip().splitlines() if line.strip()), "")
    return line.strip().strip('"').strip()


def refine_query(state: ResearchState) -> dict:
    next_round = state.get("retrieval_round", 1) + 1
    reply = llm.invoke(QUERY_PROMPTS[next_round].invoke({"question": state["question"]})).content
    query = clean_query(reply)
    return {
        "query": query,
        "retrieval_round": next_round,
        "history": [f'refine_query: round {next_round} searches for "{query}"'],
    }


print(refine_query({"question": QUESTIONS["e"]}))

### 4 State that survives the loop

Every round adds documents and claims, and nothing from an earlier round may get lost. `documents`
already has its merge rule from NB 02. `claims` was written once per run in NB 03; in a loop it is
written in every round, so it now needs one too. `merge_claims` keeps every claim already read, adds
new ones, and leaves a document that is read again with its first claim.

In [ ]:
round_one = {"E1": "0.38%", "E2": "0.38%", "E3": "0.62%"}
round_two = {"E4": "0.62%"}
print(merge_claims(round_one, round_two))
print(merge_claims(round_one, {"E3": "a different reading"})["E3"])

Mistake 3 shows what happens to the state without this rule.

### 5 The graph

The limited answer gets a node of its own, `write_limitation`, so that the stop is visible in the
graph and its text can say what was found, why the gate failed, how many rounds ran, and that no
further search will run. The path map in `add_conditional_edges` lists **every** label the router can
return.

Compared with NB 03, the contract gains three entries: `query` and `retrieval_round`, both owned by
`refine_query`, and `write_limitation` as a second owner of `draft_answer`.

In [ ]:
def limitation_text(state: ResearchState) -> str:
    result = assess(state["documents"], state["claims"])
    rounds = state.get("retrieval_round", 1)
    if result.unresolved:
        why = "qualifying sources contradict each other"
    elif result.answer:
        why = (f"{result.answer} rests on {', '.join(result.supporting)} only "
               f"(evidence score {result.evidence_score:g}, threshold {EVIDENCE_THRESHOLD:.1f})")
    else:
        why = f"no source at or above the authority floor of {AUTHORITY_FLOOR:g} states a figure"
    found = ", ".join(doc["source_id"] for doc in state["documents"]) or "none"
    notes = [f"{sid} disregarded: {reason}." for sid, reason in sorted(result.disregarded.items())]
    head = (f"Limited answer after {rounds} of {MAX_ROUNDS} search rounds: {why}, so the quality gate "
            f"is not passed. Documents found: {found}. No further search will run.")
    return " ".join([head, *notes])


def write_limitation(state: ResearchState) -> dict:
    return {"draft_answer": limitation_text(state), "history": ["write_limitation: limited answer written"]}


CONTRACT = {
    **RESEARCH_CONTRACT,
    "query": {"refine_query"},
    "retrieval_round": {"refine_query"},
    "draft_answer": {"write_answer", "write_limitation"},
}

builder = StateGraph(ResearchState)
builder.add_node("collect_primary", collect_primary)
builder.add_node("collect_secondary", collect_secondary)
builder.add_node("score", make_score(llm))
builder.add_node("refine_query", refine_query)
builder.add_node("write_answer", write_answer)
builder.add_node("write_limitation", write_limitation)
builder.add_edge(START, "collect_primary")
builder.add_edge(START, "collect_secondary")
builder.add_edge(["collect_primary", "collect_secondary"], "score")
builder.add_conditional_edges(
    "score",
    route,
    {"answer": "write_answer", "search_again": "refine_query", "limit_answer": "write_limitation"},
)
builder.add_edge("refine_query", "collect_primary")
builder.add_edge("refine_query", "collect_secondary")
builder.add_edge("write_answer", END)
builder.add_edge("write_limitation", END)
graph = builder.compile()

print(graph.get_graph().draw_mermaid())

### 6 Topic e: the loop pays off

`stream()` reports every node's update, so the run can be followed round by round. `trace` condenses
it to one line per collected search and score.

In [ ]:
def trace(question: str) -> dict:
    for update in graph.stream({"question": question}):
        for node, fields in update.items():
            for line in fields.get("history", []):
                print("  ", line)
    final = graph.invoke({"question": question})
    print(f"\n=> {final['draft_answer']}")
    return final


result_e = trace(QUESTIONS["e"])

Round 1 finds E1, E2 and E3; the copied 0.38% is below the authority floor, and 0.62% rests on E3
alone, so the score is 0.85 and the router sends the search back. The synonym query talks of a
"yearly charge of automated investment services": words that the consumer study E4 uses and the
question did not. Round 2 finds E4, the scorer reads only that one new document, the score rises to
1.65, and the router chooses `answer`.

### 7 Topic c: the loop stops on purpose

In [ ]:
result_c = trace(QUESTIONS["c"])

Three rounds, three different queries, and no new figure: the law-firm blog, the news preview and the
forum question are found again, but none of them states a requirement. After round 3 the router
chooses `limit_answer`. The answer says what is known (90%, from the circular), why it is not enough,
how many rounds ran, and that the search is over. That is a better outcome than a fourth round, and
a much better one than a confident answer from a single source.

### 8 All five questions

In [ ]:
results = {topic: graph.invoke({"question": question}) for topic, question in QUESTIONS.items()}
for topic, final in results.items():
    check_contract(graph.stream({"question": QUESTIONS[topic]}), CONTRACT)
    print(f"{topic}: {final.get('retrieval_round', 1)} round(s), evidence score {final['evidence_score']:g}")

| Topic | Rounds | Outcome |
|---|---|---|
| a | 1 | answered: CHF 342 million; the forum claim is flagged and disregarded |
| b | 1 | answered: CHF 1.2 billion |
| c | 3 | **limited answer:** 90% from one source; the search stopped at the maximum |
| d | 1 | answered: 45%; the 2023 target is outdated |
| e | 2 | **answered in round 2:** 0.62%, once the rephrased query found E4 |

This is the point where a linear chain stops being enough. A chain cannot return from the quality
gate to the search and carry along what it has found, which claims it has read, and how many rounds
it has used. A conditional edge and a state with merge rules do exactly that.

### 9 Business stop and emergency brake

Topic c used 3 search rounds, the business limit. In LangGraph's own counting (one step per group of
nodes that run together, plus the start), the run needs a `recursion_limit` of at least 10; the
default is 25, and it would only be reached if the stop condition were missing. The next cell
measures that. The two have different jobs: `MAX_ROUNDS` says "more searching will
not help", and its outcome is an answer; `recursion_limit` says "something is wrong with this graph",
and its outcome is an exception.

In [ ]:
for limit in (9, 10):
    try:
        graph.invoke({"question": QUESTIONS["c"]}, {"recursion_limit": limit})
        print(f"recursion_limit {limit}: the run finishes")
    except GraphRecursionError:
        print(f"recursion_limit {limit}: aborted by the emergency brake")

## Common mistakes

### Mistake 1: a loop without a stop condition

The router below always says `search_again`. Nothing ends the loop, so LangGraph's emergency brake
does. The limit is set to 12 here to fail fast; the stand-in nodes make no model calls.

In [ ]:
def stuck_score(state: ResearchState) -> dict:
    return {"evidence_score": 0.95}


def count_round(state: ResearchState) -> dict:
    return {"retrieval_round": state.get("retrieval_round", 1) + 1}


endless = StateGraph(ResearchState)
endless.add_node("score", stuck_score)
endless.add_node("refine_query", count_round)
endless.add_edge(START, "score")
endless.add_conditional_edges("score", lambda state: "search_again", {"search_again": "refine_query"})
endless.add_edge("refine_query", "score")
endless.compile().invoke({"question": QUESTIONS["c"]}, {"recursion_limit": 12})

*"Recursion limit of 12 reached without hitting a stop condition."* The run fails as an error instead
of ending with an answer, and the reader learns nothing about the evidence. **Fix:** a stop condition
from the domain inside the router (`MAX_ROUNDS`), so that `recursion_limit` is never what ends a run.

### Mistake 2: the router returns a label the map does not know

A router and its path map are written in two places, which makes it easy to rename one and forget
the other. Here the map was written with only the `answer` label.

In [ ]:
incomplete = StateGraph(ResearchState)
incomplete.add_node("score", stuck_score)
incomplete.add_node("write_answer", write_answer)
incomplete.add_edge(START, "score")
incomplete.add_conditional_edges("score", lambda state: "search_again", {"answer": "write_answer"})
incomplete.add_edge("write_answer", END)
incomplete.compile().invoke({"question": QUESTIONS["c"]})

`KeyError: 'search_again'`: the graph compiles, and the gap shows only when that label actually comes
up, possibly in a rare case long after testing. **Fix:** map every label the router can return, and
test each route, as `tests/test_loop.py` does.

### Mistake 3: a field written in every round, without a merge rule

This is the quietest of the three. `claims` here has no merge rule, so each round's update replaces
the previous one. The scorer still reads the earlier claims from the state before returning, so the
router decides correctly. Then the state loses them.

In [ ]:
class StateWithoutClaimMerge(TypedDict, total=False):
    question: str
    query: str
    retrieval_round: int
    documents: Annotated[list[SourceDoc], merge_documents]
    claims: dict[str, str | None]
    evidence_score: float
    history: Annotated[list[str], operator.add]


def collect_all(state: StateWithoutClaimMerge) -> dict:
    search = state.get("query") or state["question"]
    return {"documents": retrieve(search, corpus(), weights())}


def score_new_only(state: StateWithoutClaimMerge) -> dict:
    known = state.get("claims") or {}
    new = {d["source_id"]: extract_claim(llm, state["question"], d) for d in state["documents"] if d["source_id"] not in known}
    return {"claims": new, "evidence_score": assess(state["documents"], {**known, **new}).evidence_score}


def refine_without_merge(state: StateWithoutClaimMerge) -> dict:
    return refine_query(state)


forgetful = StateGraph(StateWithoutClaimMerge)
forgetful.add_node("collect", collect_all)
forgetful.add_node("score", score_new_only)
forgetful.add_node("refine_query", refine_without_merge)
forgetful.add_edge(START, "collect")
forgetful.add_edge("collect", "score")
forgetful.add_conditional_edges(
    "score", lambda state: route(state), {"answer": END, "search_again": "refine_query", "limit_answer": END}
)
forgetful.add_edge("refine_query", "collect")

forgetful_result = forgetful.compile().invoke({"question": QUESTIONS["e"]})
print("rounds:", forgetful_result.get("retrieval_round", 1), "| claims left:", sorted(forgetful_result["claims"]))
assert not assess(forgetful_result["documents"], forgetful_result["claims"]).sufficient

The router chose `answer` in round 2, but the claims of round 1 are gone. Anyone who reads the state
afterwards (the writer, a reviewer, NB 06's saved checkpoint) finds E3 without a claim and concludes
that the evidence is insufficient: the state now contradicts the decision taken on it. No error
anywhere. **Fix:** give every field that is written in more than one round a merge rule, as
`merge_claims` does.

A technical note on this example. Its nodes are written for its own state class, because LangGraph
reads a node's type hint to decide which state it belongs to. The router is a different kind of
function: read-only control logic that selects an edge and never updates the state. Its parameter is
still typed as the graph's state, and LangGraph checks that type when the conditional edge is added.
So in a graph with a different state class, or in a unit test that passes a partial dictionary, the
router is wrapped, here as `lambda state: route(state)`. That is an interface detail, not a sign
that the router has become a node.

## Prompt log

The queries in this notebook were recorded from `qwen3:8b` (temperature 0, thinking off) on
5 October 2026, for the two questions that reach a second round.

**Version 1**

```text
Rephrase the question as a search query.

Question: {question}
```

**What happened.** For topic e the model replied *Search query: "Average annual fee of robo-advisory
platforms in Switzerland 2025"*. Two problems: the reply wrapped the query in a label and quotation
marks, and, more importantly, it kept the question's own words. The search found E1, E2 and E3 again,
and not E4. Rephrasing in the same words is the same search.

**Version 2 (final, round 2)**

```text
Write a new search query for the question below. Use different words: replace the question's key
terms with synonyms and the technical terms used in the field, so that documents that describe the
same thing in other words are found too. Keep any names. Reply with the query only, on one line.

Question: {question}
```

**What changed.** Topic e: *"What is the typical yearly charge of automated investment services in
Switzerland for 2025?"*, which finds E4. Topic c: a query about "cryptocurrency holdings" in "offline
wallets", which finds the circular and the news preview again, and nothing new.

**Round 3 prompt.** *"Write a broader search query about the general subject of the question..."*
gave, for topic c, *"Regulatory requirements for crypto asset custody in cold storage under AMSA
rules"*. It finds all four topic c documents, still with one figure among them, so the stop is the
right outcome.

**Variation between calls.** Asked three times with the identical round 2 prompt for topic e, the
live model wrote "for 2025" once and "for the year 2025" twice, at temperature 0. The meaning, and
the documents found, stayed the same; the wording did not. Temperature 0 makes a model more
consistent, not deterministic, which is one more reason the tests run on recorded replies.

**Insight.** A second search only helps if it searches differently, and a model will not do that
unless asked to change the words. "Keep any names" matters too: the retrieval rule of NB 03 requires
documents to contain the question's names, and a query that dropped "Switzerland" or "AMSA" would
lose that check. The model does not decide whether to search again either. It writes
queries; the router decides.

In [ ]:
NAIVE_PROMPT = ChatPromptTemplate.from_template("Rephrase the question as a search query.\n\nQuestion: {question}")
for name, prompt in (("Version 1", NAIVE_PROMPT), ("Version 2", SYNONYM_PROMPT)):
    query = clean_query(llm.invoke(prompt.invoke({"question": QUESTIONS["e"]})).content)
    found = [doc["source_id"] for doc in retrieve(query, corpus(), weights())]
    print(f"{name}: {query}\n   finds {found}")

With `LLM_MODE=ollama`, the next cell asks the live model for the round 2 and round 3 queries of both
questions. It is skipped in scripted mode and in the tests.

In [ ]:
for topic in ("c", "e"):
    for round_number, prompt in QUERY_PROMPTS.items():
        query = clean_query(llm.invoke(prompt.invoke({"question": QUESTIONS[topic]})).content)
        print(f"{topic}, round {round_number}: {query}")

## Summary

The assistant now decides what to do after scoring. A router reads the evidence score and the round
counter and chooses one of three routes. A search round that repeats must search differently, so each
round has its own query strategy while the quality gate stays fixed. Documents and claims survive the
loop because both fields have merge rules. And the loop ends on a condition from the domain, with an
answer that states its limitation, rather than on LangGraph's emergency brake.

The loop lives in the package as `loop.py`. The next cell checks that the notebook and the package
agree. In every mode it compares what does not depend on the model: the graph's structure, the
contract, the router's decisions and the limitation text for the same state. Only in scripted mode,
where every run is identical, does it also compare complete runs: a live model can word the same
query slightly differently from one call to the next (see the prompt log), so two live runs are not
expected to match character for character.

In [ ]:
from research_assistant import loop as package_loop

package_graph = package_loop.build_loop_graph(llm)
structure = lambda g: (set(g.get_graph().nodes), {(e.source, e.target) for e in g.get_graph().edges})
assert structure(package_graph) == structure(graph)
assert package_loop.LOOP_CONTRACT == CONTRACT
assert package_loop.limitation_text(results["c"]) == limitation_text(results["c"])
if mode == "fake":
    for topic, question in QUESTIONS.items():
        assert package_graph.invoke({"question": question}) == results[topic]
for example in ({"evidence_score": 1.6}, {"evidence_score": 0.85}, {"evidence_score": 0.95, "retrieval_round": 3}):
    assert package_loop.route(example) == route(example)
print("Notebook and package agree.")

**Next: NB 05, LLM misjudgement.** So far the model has only read figures and written queries, and
the rules have judged. NB 05 asks the model to judge the evidence itself, and compares its verdicts
with the rules on the cases built into the corpus: a contradiction, an outdated source, and two
sources that are really one.

## Exercise

1. Set `MAX_ROUNDS = 2`, rebuild the graph and run topics c and e. Which answers change, and which do
   not? What would you lose and gain with this setting in practice?
2. Write a `refine_query` that keeps the question unchanged instead of rephrasing it. Predict the
   number of rounds for topic e before running it, then check. What does the result say about loops
   that repeat the same action?